# Create Synthetic Dataset of 2000 rows 

In [14]:
"""
Crop Water Requirement (CWR) Dataset Generator for Paddy Cultivation
====================================================================
This script generates synthetic training data for ML-based irrigation prediction
using expert-validated parameters and agricultural principles.

Author: [Yeran Fernando]
Project: ML-Based Predictive Irrigation System for Paddy Cultivation
Date: March 2026
"""

import numpy as np
import pandas as pd
from datetime import datetime

# ============================================================================
# EXPERT PARAMETERS FROM QUESTIONNAIRE
# ============================================================================

class ExpertParameters:
    """
    All parameters extracted from agricultural expert questionnaire responses.
    Each parameter includes the updated question number reference.

    Updated Stage Mapping (March 2026):
        1 = Land Preparation
        2 = Seedling
        3 = Vegetative
        4 = Reproductive
        5 = Ripening
    """

    # SECTION 1: FIELD WATER DEPTH REQUIREMENTS (Q1-Q4)
    # All values converted to mm for consistency
    # Note: Stage-specific MAX depth is now used as the harmful threshold
    # Q5 (global harmful depth) removed - each stage defines its own safe maximum

    # Q1: Land Preparation stage water depth
    LAND_PREP_IDEAL_DEPTH = 150  # mm (15cm — puddling and soil softening)
    LAND_PREP_MIN_DEPTH   = 50   # mm (5cm)
    LAND_PREP_MAX_DEPTH   = 200  # mm (20cm)

    # Q2: Seedling stage water depth
    SEEDLING_IDEAL_DEPTH = 30    # mm (3cm)
    SEEDLING_MIN_DEPTH   = 10    # mm (1cm)
    SEEDLING_MAX_DEPTH   = 50    # mm (5cm)

    # Q3: Vegetative stage water depth
    VEGETATIVE_IDEAL_DEPTH = 40  # mm (4cm)
    VEGETATIVE_MIN_DEPTH   = 10  # mm (1cm)
    VEGETATIVE_MAX_DEPTH   = 60  # mm (6cm)

    # Q4: Reproductive stage water depth
    REPRODUCTIVE_IDEAL_DEPTH = 150   # mm (15cm — critical flooding stage)
    REPRODUCTIVE_MIN_DEPTH   = 50    # mm (5cm)
    REPRODUCTIVE_MAX_DEPTH   = 170   # mm (17cm)

    # SECTION 1B: SOIL MOISTURE REQUIREMENTS (Q5)
    # Note: Q7 (stress threshold) and Q8 (saturation threshold) removed
    # RIPENING_MIN_MOISTURE now serves as the critical stress threshold

    # Q5: Ripening stage soil moisture (%)
    RIPENING_IDEAL_MOISTURE = 90   # % (expert answer: ideal 90%)
    RIPENING_MIN_MOISTURE   = 85   # % — also used as critical stress threshold
    RIPENING_MAX_MOISTURE   = 95   # %

    # Assumed: Saturated soil moisture for flooded stages (literature value)
    FLOODED_SOIL_SATURATION = 95   # % (soil under standing water is nearly saturated)

    # SECTION 2: CROP GROWTH STAGES (Q6)
    CROP_STAGE_DURATIONS = {
        1: 10.5,   # Land Preparation: 7-14 days (average)
        2: 17.5,   # Seedling: 15-20 days (average)
        3: 30.0,   # Vegetative: 25-35 days (average)
        4: 32.5,   # Reproductive: 30-35 days (average)
        5: 27.5    # Ripening: 25-30 days (average)
    }

    # SECTION 3: WATER REQUIREMENTS AND TANK LEVELS (Q7-Q8)

    # Q7: Minimum tank level for safe irrigation
    TANK_MIN_THRESHOLD = 27.5    # % (average of 25-30%)

    # Q8: Priority stages when tank level is low
    # Stage 4 (Reproductive) is highest priority, then Stage 3 (Vegetative)
    PRIORITY_STAGES = [4, 3]

    # SECTION 4: RAINFALL IMPACT (Q9-Q10)

    # Q9: Rainfall prediction time window
    RAINFALL_PREDICTION_WINDOW = 24   # hours

    # Q10: Skip irrigation threshold (heavy rain)
    RAINFALL_SKIP_THRESHOLD   = 10    # mm (>10mm = skip entirely)

    # Q10: Moderate rainfall range — reduce irrigation
    RAINFALL_REDUCE_THRESHOLD = 5     # mm (5-10mm = reduce to 40%)
    RAINFALL_REDUCTION_FACTOR = 0.4   # 40% of normal (middle of 30-50%)

    # SECTION 5: EVAPOTRANSPIRATION (Q11-Q14)

    # Q13: Normal ET rate in Sri Lankan tropical conditions
    ET_NORMAL_MIN = 5    # mm/day
    ET_NORMAL_MAX = 7    # mm/day

    # Q14: High ET threshold requiring extra irrigation
    HIGH_ET_THRESHOLD = 8.5   # mm/day (average of 8-9 mm/day)

    # Upper bound for high ET in Sri Lankan conditions (literature)
    ET_HIGH_MAX = 10   # mm/day

    # Q11: Hot days water increase factor
    HOT_DAY_INCREASE = 0.175  # 17.5% (average of 15-20%)

    # Q12: Cool days water decrease factor
    COOL_DAY_DECREASE = 0.125  # 12.5% (average of 10-15%)

    # ET high-probability weights per stage
    # Source: Agronomic literature
    # Reproductive = peak canopy coverage = highest ET
    # Vegetative = growing canopy = rising ET
    # Land Preparation = no plants = soil evaporation only
    ET_HIGH_PROB = {
        1: 0.05,   # Land Preparation — soil evaporation only, minimal ET
        2: 0.20,   # Seedling — small plants, low ET
        3: 0.40,   # Vegetative — growing canopy, rising ET
        4: 0.50,   # Reproductive — full canopy, peak ET
        5: 0.10    # Ripening — field draining, declining ET
    }

    # SECTION 6: DECISION RULES (Q15-Q18)

    # Q16: Extra water buffer for high ET days
    HIGH_ET_BUFFER = 0.15   # 15% (average of 10-20%)

    # Q17: Minimum water amount to justify opening the irrigation gate
    # Below this, wait for deficit to accumulate (unless critical situation)
    MIN_CWR_THRESHOLD = 2   # mm (expert answer: 2mm minimum)

    # Q18: Preferred irrigation timing (reference only, not used in calculation)
    PREFERRED_IRRIGATION_TIME = "6-8 AM"

    # ADDITIONAL PARAMETERS (agronomic literature — not from questionnaire)

    # Typical percolation/seepage rate for paddy fields
    PERCOLATION_RATE = 3   # mm/day (typical for clay-loam paddy soil)

    # Dried field detection threshold
    DRIED_FIELD_THRESHOLD = 0.5   # cm (0.5cm or less = field considered dried)

    # Moisture to water conversion factor
    # Formula: (bulk_density / water_density) x root_zone_depth / 100
    # = (1.3 g/cm3 / 1.0 g/cm3) x 200mm / 100 = 2.6 mm per 1%
    # Root zone: 200mm (FAO paddy standard)
    # Bulk density: 1.3 g/cm3 (Sri Lankan irrigated clay paddy)
    # Ref: Rathnayake et al. (UWU), FAO Irrigation Manual (1992)
    MOISTURE_TO_WATER_FACTOR = 2.6   # mm per 1% soil moisture


# Stage name mapping for readable output
STAGE_NAMES = {
    1: 'Land Preparation',
    2: 'Seedling',
    3: 'Vegetative',
    4: 'Reproductive',
    5: 'Ripening'
}

# Create instance for easy access
params = ExpertParameters()


# ============================================================================
# CWR CALCULATION FUNCTION
# ============================================================================

def calculate_CWR(water_depth_cm, soil_moisture_pct, tank_level_pct,
                  ET_mm, rainfall_mm, crop_stage):
    """
    Calculate Crop Water Requirement (CWR) using water balance equation
    with expert-validated decision rules and constraints.

    Parameters:
    -----------
    water_depth_cm    : float — Current standing water depth in field (cm)
    soil_moisture_pct : float — Current soil moisture (%)
    tank_level_pct    : float — Current tank water level (%)
    ET_mm             : float — Evapotranspiration rate (mm/day)
    rainfall_mm       : float — Predicted rainfall for next 24 hours (mm)
    crop_stage        : int   — 1=Land Prep, 2=Seedling, 3=Vegetative,
                                4=Reproductive, 5=Ripening

    Returns:
    --------
    float : CWR in mm (amount of water to add via irrigation)
    """

    # ========================================================================
    # STEP 1: EXTRACT STAGE-SPECIFIC PARAMETERS (Q1-Q5)
    # ========================================================================

    if crop_stage == 1:    # Land Preparation
        target_depth = params.LAND_PREP_IDEAL_DEPTH
        min_depth    = params.LAND_PREP_MIN_DEPTH
        max_depth    = params.LAND_PREP_MAX_DEPTH
    elif crop_stage == 2:  # Seedling
        target_depth = params.SEEDLING_IDEAL_DEPTH
        min_depth    = params.SEEDLING_MIN_DEPTH
        max_depth    = params.SEEDLING_MAX_DEPTH
    elif crop_stage == 3:  # Vegetative
        target_depth = params.VEGETATIVE_IDEAL_DEPTH
        min_depth    = params.VEGETATIVE_MIN_DEPTH
        max_depth    = params.VEGETATIVE_MAX_DEPTH
    elif crop_stage == 4:  # Reproductive
        target_depth = params.REPRODUCTIVE_IDEAL_DEPTH
        min_depth    = params.REPRODUCTIVE_MIN_DEPTH
        max_depth    = params.REPRODUCTIVE_MAX_DEPTH
    else:                  # Ripening (stage 5)
        target_moisture = params.RIPENING_IDEAL_MOISTURE
        min_moisture    = params.RIPENING_MIN_MOISTURE
        max_moisture    = params.RIPENING_MAX_MOISTURE

    # ========================================================================
    # STEP 2: CALCULATE BASE CWR USING WATER BALANCE
    # ========================================================================

    critical_situation = False   # Flag for urgent irrigation needs

    if crop_stage <= 4:   # FLOODED STAGES (Land Prep to Reproductive)

        current_depth_mm = water_depth_cm * 10   # Convert cm to mm

        # Below stage minimum = critical situation (Q1-Q4 min values)
        if current_depth_mm < min_depth:
            critical_situation = True

        if water_depth_cm < params.DRIED_FIELD_THRESHOLD:
            # DRIED FIELD: need to saturate soil first, then add standing water

            # Water needed to saturate soil from current moisture level
            if soil_moisture_pct < params.FLOODED_SOIL_SATURATION:
                soil_moisture_deficit = ((params.FLOODED_SOIL_SATURATION - soil_moisture_pct)
                                         * params.MOISTURE_TO_WATER_FACTOR)
            else:
                soil_moisture_deficit = 0

            # Full target standing water depth + soil saturation + losses
            base_CWR = (soil_moisture_deficit + target_depth +
                        ET_mm + params.PERCOLATION_RATE - rainfall_mm)

        else:
            # NORMAL FLOODED CONDITION — soil already saturated
            depth_deficit = max(0, target_depth - current_depth_mm)
            base_CWR = (depth_deficit + ET_mm + params.PERCOLATION_RATE - rainfall_mm)

    else:   # RIPENING STAGE (Stage 5) — drained field

        # Below minimum moisture = critical situation (Q5 min value)
        if soil_moisture_pct < params.RIPENING_MIN_MOISTURE:
            critical_situation = True

        moisture_deficit = max(0, target_moisture - soil_moisture_pct)
        water_needed_for_moisture = moisture_deficit * params.MOISTURE_TO_WATER_FACTOR
        base_CWR = (water_needed_for_moisture + ET_mm +
                    params.PERCOLATION_RATE - rainfall_mm)

    # ========================================================================
    # STEP 3: APPLY RAINFALL DECISION RULES (Q10)
    # ========================================================================

    if rainfall_mm > params.RAINFALL_SKIP_THRESHOLD:
        return 0   # Heavy rain — skip irrigation entirely

    elif rainfall_mm > params.RAINFALL_REDUCE_THRESHOLD:
        base_CWR = base_CWR * params.RAINFALL_REDUCTION_FACTOR   # Moderate rain — reduce to 40%

    # ========================================================================
    # STEP 4: CRITICAL SITUATION BOOST (Q1-Q4 min, Q5 min moisture)
    # ========================================================================

    if critical_situation:
        base_CWR = base_CWR * 1.3   # Boost 30% — restore conditions quickly

    # ========================================================================
    # STEP 5: TANK LEVEL CHECK (Q7, Q8)
    # ========================================================================
    # Q7: Tank min threshold is a binary operational gate —
    # below minimum, delivery pressure is unreliable regardless of CWR.
    # Above minimum, full CWR is delivered (tank level does not
    # gradually reduce irrigation — the CWR formula already scales output).
    
    if tank_level_pct >= params.TANK_MIN_THRESHOLD:
        # Sufficient tank level — deliver full calculated CWR
        tank_factor = 1.0
    
    else:
        # BELOW MINIMUM — apply stage priority (Q8)
        if crop_stage in params.PRIORITY_STAGES or critical_situation:
            tank_factor = 0.5   # Priority stages (Reproductive, Vegetative) get 50%
        else:
            tank_factor = 0.2   # Non-priority stages get 20%
    
    adjusted_CWR = base_CWR * tank_factor


    # ========================================================================
    # STEP 6: HIGH ET ADJUSTMENT (Q11, Q14, Q16)
    # ========================================================================

    if ET_mm > params.HIGH_ET_THRESHOLD:
        adjusted_CWR = adjusted_CWR * (1 + params.HIGH_ET_BUFFER)   # Add 15% buffer

    # ========================================================================
    # STEP 7: SAFETY CONSTRAINTS
    # ========================================================================

    if crop_stage <= 4:   # Flooded stages
        current_depth_mm = water_depth_cm * 10

        # Stage max depth IS the harmful threshold (Q5 removed, Q1-Q4 max used directly)
        max_addable = max_depth - current_depth_mm

        if max_addable < 0:
            return 0   # Field already over-flooded — no irrigation

        adjusted_CWR = min(adjusted_CWR, max_addable)

    # Q17: Minimum threshold check — with critical situation override
    if adjusted_CWR < params.MIN_CWR_THRESHOLD:
        if critical_situation:
            pass   # Emergency override — irrigate despite small amount
        else:
            adjusted_CWR = 0   # Too small — wait for deficit to accumulate

    # ========================================================================
    # STEP 8: FINAL OUTPUT
    # ========================================================================

    return max(0, adjusted_CWR)


# ============================================================================
# DATASET GENERATION FUNCTION
# ============================================================================

def generate_dataset(num_samples=1000, random_seed=42):
    """
    Generate synthetic training dataset for CWR prediction model.

    Parameters:
    -----------
    num_samples  : int — Number of training samples (default: 1000)
    random_seed  : int — Seed for reproducibility (default: 42)

    Returns:
    --------
    pandas.DataFrame : Dataset with input features and CWR labels
    """

    np.random.seed(random_seed)

    print(f"Generating {num_samples} training samples...")
    print("=" * 60)

    data = []
    samples_per_stage = num_samples // 5

    # ========================================================================
    # GENERATE SCENARIOS FOR EACH CROP STAGE
    # ========================================================================

    for stage in range(1, 6):   # Stages 1-5

        print(f"\nGenerating Stage {stage} ({STAGE_NAMES[stage]}) scenarios...")

        if stage <= 4:   # FLOODED STAGES

            if stage == 1:
                depth_min, depth_ideal, depth_max = (params.LAND_PREP_MIN_DEPTH / 10,
                                                      params.LAND_PREP_IDEAL_DEPTH / 10,
                                                      params.LAND_PREP_MAX_DEPTH / 10)
            elif stage == 2:
                depth_min, depth_ideal, depth_max = (params.SEEDLING_MIN_DEPTH / 10,
                                                      params.SEEDLING_IDEAL_DEPTH / 10,
                                                      params.SEEDLING_MAX_DEPTH / 10)
            elif stage == 3:
                depth_min, depth_ideal, depth_max = (params.VEGETATIVE_MIN_DEPTH / 10,
                                                      params.VEGETATIVE_IDEAL_DEPTH / 10,
                                                      params.VEGETATIVE_MAX_DEPTH / 10)
            else:   # stage == 4
                depth_min, depth_ideal, depth_max = (params.REPRODUCTIVE_MIN_DEPTH / 10,
                                                      params.REPRODUCTIVE_IDEAL_DEPTH / 10,
                                                      params.REPRODUCTIVE_MAX_DEPTH / 10)

            # Water depth distribution
            water_depths = []
            water_depths.extend(np.random.uniform(0, 0.4,
                                int(samples_per_stage * 0.20)))                   # 20% dried out
            water_depths.extend(np.random.uniform(depth_min, depth_ideal,
                                int(samples_per_stage * 0.30)))                   # 30% below ideal
            water_depths.extend(np.random.uniform(depth_ideal * 0.8, depth_ideal * 1.2,
                                int(samples_per_stage * 0.30)))                   # 30% around ideal
            water_depths.extend(np.random.uniform(depth_ideal, depth_max,
                                int(samples_per_stage * 0.20)))                   # 20% above ideal

            # Soil moisture — high when flooded, lower when dried
            soil_moistures = []
            for wd in water_depths:
                if wd < 0.5:
                    soil_moistures.append(np.random.uniform(50, 85))    # Dried field
                else:
                    soil_moistures.append(np.random.uniform(90, 100))   # Flooded field

        else:   # RIPENING STAGE (Stage 5) — drained field

            water_depths = np.zeros(samples_per_stage)   # No standing water

            soil_moistures = []
            soil_moistures.extend(np.random.uniform(params.RIPENING_MIN_MOISTURE,
                                                    params.RIPENING_IDEAL_MOISTURE,
                                                    int(samples_per_stage * 0.25)))   # 25% below ideal
            soil_moistures.extend(np.random.uniform(45, params.RIPENING_MIN_MOISTURE,
                                                    int(samples_per_stage * 0.25)))   # 25% critically dry
            soil_moistures.extend(np.random.uniform(params.RIPENING_IDEAL_MOISTURE * 0.9,
                                                    params.RIPENING_IDEAL_MOISTURE * 1.1,
                                                    int(samples_per_stage * 0.30)))   # 30% around ideal
            soil_moistures.extend(np.random.uniform(params.RIPENING_IDEAL_MOISTURE,
                                                    params.RIPENING_MAX_MOISTURE,
                                                    int(samples_per_stage * 0.20)))   # 20% above ideal

        water_depths   = list(water_depths)[:samples_per_stage]
        soil_moistures = list(soil_moistures)[:samples_per_stage]

        # Tank levels: uniform distribution
        tank_levels = np.random.uniform(15, 100, samples_per_stage)

        # ET rates: stage-specific distribution from agronomic literature
        high_prob   = params.ET_HIGH_PROB[stage]
        low_prob    = 0.10
        normal_prob = 1.0 - high_prob - low_prob

        ET_rates = []
        ET_rates.extend(np.random.uniform(params.ET_NORMAL_MIN, params.ET_NORMAL_MAX,
                                         int(samples_per_stage * normal_prob)))              # normal
        ET_rates.extend(np.random.uniform(params.HIGH_ET_THRESHOLD, params.ET_HIGH_MAX,
                                         int(samples_per_stage * high_prob)))                # high
        ET_rates.extend(np.random.uniform(4, params.ET_NORMAL_MIN,
                                         int(samples_per_stage * low_prob)))                 # low
        ET_rates = np.array(ET_rates[:samples_per_stage])
        np.random.shuffle(ET_rates)

        # Rainfall distribution (Q10)
        rainfalls = []
        rainfalls.extend(np.zeros(int(samples_per_stage * 0.50)))                           # 50% no rain
        rainfalls.extend(np.random.uniform(0.1, 5,  int(samples_per_stage * 0.25)))         # 25% light
        rainfalls.extend(np.random.uniform(5,   10, int(samples_per_stage * 0.15)))         # 15% moderate
        rainfalls.extend(np.random.uniform(10,  20, int(samples_per_stage * 0.10)))         # 10% heavy
        rainfalls = np.array(rainfalls[:samples_per_stage])
        np.random.shuffle(rainfalls)

        # Generate samples for this stage
        for i in range(samples_per_stage):

            water_depth   = water_depths[i]
            soil_moisture = soil_moistures[i]
            tank_level    = tank_levels[i]
            ET            = ET_rates[i]
            rainfall      = rainfalls[i]

            CWR = calculate_CWR(water_depth, soil_moisture, tank_level,
                                ET, rainfall, stage)

            data.append({
                'Water_Depth_cm':        round(water_depth, 2),
                'Soil_Moisture_%':       round(soil_moisture, 1),
                'Tank_Level_%':          round(tank_level, 1),
                'ET_mm_day':             round(ET, 2),
                'Rainfall_Predicted_mm': round(rainfall, 2),
                'Crop_Stage':            stage,
                'CWR_mm':                round(CWR, 2)
            })

        print(f"  Generated {samples_per_stage} samples for Stage {stage} ({STAGE_NAMES[stage]})")

    # ========================================================================
    # CREATE DATAFRAME
    # ========================================================================

    df = pd.DataFrame(data)
    df = df.sample(frac=1, random_state=random_seed).reset_index(drop=True)

    print("\n" + "=" * 60)
    print("Dataset generation complete!")
    print(f"Total samples: {len(df)}")
    print("\nDataset statistics:")
    print(df.describe())

    print("\nCWR distribution:")
    print(f"  Zero CWR (skip irrigation): {(df['CWR_mm'] == 0).sum()} samples ({(df['CWR_mm'] == 0).sum()/len(df)*100:.1f}%)")
    print(f"  Low CWR   (5-15mm):  {((df['CWR_mm'] >= 5)  & (df['CWR_mm'] < 15)).sum()} samples")
    print(f"  Moderate  (15-30mm): {((df['CWR_mm'] >= 15) & (df['CWR_mm'] < 30)).sum()} samples")
    print(f"  High CWR  (30-45mm): {((df['CWR_mm'] >= 30) & (df['CWR_mm'] <= 45)).sum()} samples")

    print("\nSamples per crop stage:")
    stage_counts = df['Crop_Stage'].value_counts().sort_index()
    for s, count in stage_counts.items():
        print(f"  Stage {s} ({STAGE_NAMES[s]}): {count} samples")

    # ========================================================================
    # MONOTONICITY SANITY CHECKS
    # ========================================================================

    print("\n" + "=" * 60)
    print("RUNNING MONOTONICITY SANITY CHECKS...")
    print("=" * 60)

    all_passed = True

    # Use Stage 3 (Vegetative) as reference — flooded, high ET stage
    ref = df[df['Crop_Stage'] == 3].copy()

    corr_rain = ref['CWR_mm'].corr(ref['Rainfall_Predicted_mm'])
    if corr_rain < 0:
        print(f"  Rainfall vs CWR (Stage 3 Vegetative): Negative ({corr_rain:.3f}) — PASSED")
    else:
        print(f"  Rainfall vs CWR (Stage 3 Vegetative): Expected negative ({corr_rain:.3f}) — FAILED")
        all_passed = False

    corr_et = ref['CWR_mm'].corr(ref['ET_mm_day'])
    if corr_et > 0:
        print(f"  ET vs CWR (Stage 3 Vegetative): Positive ({corr_et:.3f}) — PASSED")
    else:
        print(f"  ET vs CWR (Stage 3 Vegetative): Expected positive ({corr_et:.3f}) — FAILED")
        all_passed = False

    ref_s5  = df[df['Crop_Stage'] == 5].copy()
    corr_sm = ref_s5['CWR_mm'].corr(ref_s5['Soil_Moisture_%'])
    if corr_sm < 0:
        print(f"  Soil Moisture vs CWR (Stage 5 Ripening): Negative ({corr_sm:.3f}) — PASSED")
    else:
        print(f"  Soil Moisture vs CWR (Stage 5 Ripening): Expected negative ({corr_sm:.3f}) — FAILED")
        all_passed = False

    heavy_rain     = df[df['Rainfall_Predicted_mm'] > params.RAINFALL_SKIP_THRESHOLD]
    non_zero_heavy = (heavy_rain['CWR_mm'] > 0).sum()
    if non_zero_heavy == 0:
        print(f"  Heavy rain (>10mm) CWR=0: All {len(heavy_rain)} samples correct — PASSED")
    else:
        print(f"  Heavy rain (>10mm) CWR=0: {non_zero_heavy} samples have CWR>0 — FAILED")
        all_passed = False

    negative_cwr = (df['CWR_mm'] < 0).sum()
    if negative_cwr == 0:
        print(f"  No negative CWR values — PASSED")
    else:
        print(f"  Found {negative_cwr} negative CWR values — FAILED")
        all_passed = False

    print("\n" + "=" * 60)
    if all_passed:
        print("ALL SANITY CHECKS PASSED — Dataset is agronomically consistent")
    else:
        print("SOME CHECKS FAILED — Review CWR formula before training")
    print("=" * 60)

    return df


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":

    print("\n" + "=" * 60)
    print("CWR DATASET GENERATOR FOR PADDY CULTIVATION")
    print("Machine Learning-Based Predictive Irrigation System")
    print("=" * 60)

    dataset = generate_dataset(num_samples=2000, random_seed=42)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename  = f"paddy_CWR_dataset_{timestamp}.csv"
    dataset.to_csv(filename, index=False)
    print(f"\nDataset saved to: {filename}")

    print("\nFirst 10 samples:")
    print(dataset.head(10).to_string())

    print("\n" + "=" * 60)
    print("EXAMPLE CWR CALCULATIONS")
    print("=" * 60)

    examples = [
        {
            'desc': 'Land Preparation — Normal flooding',
            'water_depth': 5.0, 'soil_moisture': 93, 'tank': 70,
            'ET': 4, 'rain': 0, 'stage': 1
        },
        {
            'desc': 'Seedling — Adequate water',
            'water_depth': 4.0, 'soil_moisture': 95, 'tank': 70,
            'ET': 5, 'rain': 0, 'stage': 2
        },
        {
            'desc': 'Vegetative — Dried field, critical',
            'water_depth': 0.2, 'soil_moisture': 55, 'tank': 80,
            'ET': 7, 'rain': 0, 'stage': 3
        },
        {
            'desc': 'Reproductive — Heavy rain coming',
            'water_depth': 7.0, 'soil_moisture': 96, 'tank': 75,
            'ET': 6, 'rain': 12, 'stage': 4
        },
        {
            'desc': 'Ripening — Dry soil, critical',
            'water_depth': 0, 'soil_moisture': 48, 'tank': 65,
            'ET': 5, 'rain': 0, 'stage': 5
        },
        {
            'desc': 'Vegetative — High ET, low tank',
            'water_depth': 5.5, 'soil_moisture': 93, 'tank': 35,
            'ET': 9, 'rain': 0, 'stage': 3
        }
    ]

    for ex in examples:
        CWR = calculate_CWR(ex['water_depth'], ex['soil_moisture'], ex['tank'],
                            ex['ET'], ex['rain'], ex['stage'])
        print(f"\n{ex['desc']}:")
        print(f"  Inputs: Depth={ex['water_depth']}cm, Moisture={ex['soil_moisture']}%, "
              f"Tank={ex['tank']}%, ET={ex['ET']}mm, Rain={ex['rain']}mm, "
              f"Stage={ex['stage']} ({STAGE_NAMES[ex['stage']]})")
        print(f"  CWR Output: {CWR:.2f} mm")

    print("\n" + "=" * 60)
    print("Script execution completed successfully!")
    print("=" * 60 + "\n")


CWR DATASET GENERATOR FOR PADDY CULTIVATION
Machine Learning-Based Predictive Irrigation System
Generating 2000 training samples...

Generating Stage 1 (Land Preparation) scenarios...
  Generated 400 samples for Stage 1 (Land Preparation)

Generating Stage 2 (Seedling) scenarios...
  Generated 400 samples for Stage 2 (Seedling)

Generating Stage 3 (Vegetative) scenarios...
  Generated 400 samples for Stage 3 (Vegetative)

Generating Stage 4 (Reproductive) scenarios...
  Generated 400 samples for Stage 4 (Reproductive)

Generating Stage 5 (Ripening) scenarios...
  Generated 400 samples for Stage 5 (Ripening)

Dataset generation complete!
Total samples: 2000

Dataset statistics:
       Water_Depth_cm  Soil_Moisture_%  Tank_Level_%    ET_mm_day  \
count     2000.000000      2000.000000   2000.000000  2000.000000   
mean         5.433135        88.349400     57.222250     6.666695   
std          6.130906        12.463527     25.071412     1.643597   
min          0.000000        45.10000

# Create Synthetic Dataset of 5000 rows 

In [15]:
"""
Crop Water Requirement (CWR) Dataset Generator for Paddy Cultivation
====================================================================
This script generates synthetic training data for ML-based irrigation prediction
using expert-validated parameters and agricultural principles.

Author: [Yeran Fernando]
Project: ML-Based Predictive Irrigation System for Paddy Cultivation
Date: March 2026
"""

import numpy as np
import pandas as pd
from datetime import datetime

# ============================================================================
# EXPERT PARAMETERS FROM QUESTIONNAIRE
# ============================================================================

class ExpertParameters:
    """
    All parameters extracted from agricultural expert questionnaire responses.
    Each parameter includes the updated question number reference.

    Updated Stage Mapping (March 2026):
        1 = Land Preparation
        2 = Seedling
        3 = Vegetative
        4 = Reproductive
        5 = Ripening
    """

    # SECTION 1: FIELD WATER DEPTH REQUIREMENTS (Q1-Q4)
    # All values converted to mm for consistency
    # Note: Stage-specific MAX depth is now used as the harmful threshold
    # Q5 (global harmful depth) removed - each stage defines its own safe maximum

    # Q1: Land Preparation stage water depth
    LAND_PREP_IDEAL_DEPTH = 150  # mm (15cm — puddling and soil softening)
    LAND_PREP_MIN_DEPTH   = 50   # mm (5cm)
    LAND_PREP_MAX_DEPTH   = 200  # mm (20cm)

    # Q2: Seedling stage water depth
    SEEDLING_IDEAL_DEPTH = 30    # mm (3cm)
    SEEDLING_MIN_DEPTH   = 10    # mm (1cm)
    SEEDLING_MAX_DEPTH   = 50    # mm (5cm)

    # Q3: Vegetative stage water depth
    VEGETATIVE_IDEAL_DEPTH = 40  # mm (4cm)
    VEGETATIVE_MIN_DEPTH   = 10  # mm (1cm)
    VEGETATIVE_MAX_DEPTH   = 60  # mm (6cm)

    # Q4: Reproductive stage water depth
    REPRODUCTIVE_IDEAL_DEPTH = 150   # mm (15cm — critical flooding stage)
    REPRODUCTIVE_MIN_DEPTH   = 50    # mm (5cm)
    REPRODUCTIVE_MAX_DEPTH   = 170   # mm (17cm)

    # SECTION 1B: SOIL MOISTURE REQUIREMENTS (Q5)
    # Note: Q7 (stress threshold) and Q8 (saturation threshold) removed
    # RIPENING_MIN_MOISTURE now serves as the critical stress threshold

    # Q5: Ripening stage soil moisture (%)
    RIPENING_IDEAL_MOISTURE = 90   # % (expert answer: ideal 90%)
    RIPENING_MIN_MOISTURE   = 85   # % — also used as critical stress threshold
    RIPENING_MAX_MOISTURE   = 95   # %

    # Assumed: Saturated soil moisture for flooded stages (literature value)
    FLOODED_SOIL_SATURATION = 95   # % (soil under standing water is nearly saturated)

    # SECTION 2: CROP GROWTH STAGES (Q6)
    CROP_STAGE_DURATIONS = {
        1: 10.5,   # Land Preparation: 7-14 days (average)
        2: 17.5,   # Seedling: 15-20 days (average)
        3: 30.0,   # Vegetative: 25-35 days (average)
        4: 32.5,   # Reproductive: 30-35 days (average)
        5: 27.5    # Ripening: 25-30 days (average)
    }

    # SECTION 3: WATER REQUIREMENTS AND TANK LEVELS (Q7-Q8)

    # Q7: Minimum tank level for safe irrigation
    TANK_MIN_THRESHOLD = 27.5    # % (average of 25-30%)

    # Q8: Priority stages when tank level is low
    # Stage 4 (Reproductive) is highest priority, then Stage 3 (Vegetative)
    PRIORITY_STAGES = [4, 3]

    # SECTION 4: RAINFALL IMPACT (Q9-Q10)

    # Q9: Rainfall prediction time window
    RAINFALL_PREDICTION_WINDOW = 24   # hours

    # Q10: Skip irrigation threshold (heavy rain)
    RAINFALL_SKIP_THRESHOLD   = 10    # mm (>10mm = skip entirely)

    # Q10: Moderate rainfall range — reduce irrigation
    RAINFALL_REDUCE_THRESHOLD = 5     # mm (5-10mm = reduce to 40%)
    RAINFALL_REDUCTION_FACTOR = 0.4   # 40% of normal (middle of 30-50%)

    # SECTION 5: EVAPOTRANSPIRATION (Q11-Q14)

    # Q13: Normal ET rate in Sri Lankan tropical conditions
    ET_NORMAL_MIN = 5    # mm/day
    ET_NORMAL_MAX = 7    # mm/day

    # Q14: High ET threshold requiring extra irrigation
    HIGH_ET_THRESHOLD = 8.5   # mm/day (average of 8-9 mm/day)

    # Upper bound for high ET in Sri Lankan conditions (literature)
    ET_HIGH_MAX = 10   # mm/day

    # Q11: Hot days water increase factor
    HOT_DAY_INCREASE = 0.175  # 17.5% (average of 15-20%)

    # Q12: Cool days water decrease factor
    COOL_DAY_DECREASE = 0.125  # 12.5% (average of 10-15%)

    # ET high-probability weights per stage
    # Source: Agronomic literature
    # Reproductive = peak canopy coverage = highest ET
    # Vegetative = growing canopy = rising ET
    # Land Preparation = no plants = soil evaporation only
    ET_HIGH_PROB = {
        1: 0.05,   # Land Preparation — soil evaporation only, minimal ET
        2: 0.20,   # Seedling — small plants, low ET
        3: 0.40,   # Vegetative — growing canopy, rising ET
        4: 0.50,   # Reproductive — full canopy, peak ET
        5: 0.10    # Ripening — field draining, declining ET
    }

    # SECTION 6: DECISION RULES (Q15-Q18)

    # Q16: Extra water buffer for high ET days
    HIGH_ET_BUFFER = 0.15   # 15% (average of 10-20%)

    # Q17: Minimum water amount to justify opening the irrigation gate
    # Below this, wait for deficit to accumulate (unless critical situation)
    MIN_CWR_THRESHOLD = 2   # mm (expert answer: 2mm minimum)

    # Q18: Preferred irrigation timing (reference only, not used in calculation)
    PREFERRED_IRRIGATION_TIME = "6-8 AM"

    # ADDITIONAL PARAMETERS (agronomic literature — not from questionnaire)

    # Typical percolation/seepage rate for paddy fields
    PERCOLATION_RATE = 3   # mm/day (typical for clay-loam paddy soil)

    # Dried field detection threshold
    DRIED_FIELD_THRESHOLD = 0.5   # cm (0.5cm or less = field considered dried)

    # Moisture to water conversion factor
    # Formula: (bulk_density / water_density) x root_zone_depth / 100
    # = (1.3 g/cm3 / 1.0 g/cm3) x 200mm / 100 = 2.6 mm per 1%
    # Root zone: 200mm (FAO paddy standard)
    # Bulk density: 1.3 g/cm3 (Sri Lankan irrigated clay paddy)
    # Ref: Rathnayake et al. (UWU), FAO Irrigation Manual (1992)
    MOISTURE_TO_WATER_FACTOR = 2.6   # mm per 1% soil moisture


# Stage name mapping for readable output
STAGE_NAMES = {
    1: 'Land Preparation',
    2: 'Seedling',
    3: 'Vegetative',
    4: 'Reproductive',
    5: 'Ripening'
}

# Create instance for easy access
params = ExpertParameters()


# ============================================================================
# CWR CALCULATION FUNCTION
# ============================================================================

def calculate_CWR(water_depth_cm, soil_moisture_pct, tank_level_pct,
                  ET_mm, rainfall_mm, crop_stage):
    """
    Calculate Crop Water Requirement (CWR) using water balance equation
    with expert-validated decision rules and constraints.

    Parameters:
    -----------
    water_depth_cm    : float — Current standing water depth in field (cm)
    soil_moisture_pct : float — Current soil moisture (%)
    tank_level_pct    : float — Current tank water level (%)
    ET_mm             : float — Evapotranspiration rate (mm/day)
    rainfall_mm       : float — Predicted rainfall for next 24 hours (mm)
    crop_stage        : int   — 1=Land Prep, 2=Seedling, 3=Vegetative,
                                4=Reproductive, 5=Ripening

    Returns:
    --------
    float : CWR in mm (amount of water to add via irrigation)
    """

    # ========================================================================
    # STEP 1: EXTRACT STAGE-SPECIFIC PARAMETERS (Q1-Q5)
    # ========================================================================

    if crop_stage == 1:    # Land Preparation
        target_depth = params.LAND_PREP_IDEAL_DEPTH
        min_depth    = params.LAND_PREP_MIN_DEPTH
        max_depth    = params.LAND_PREP_MAX_DEPTH
    elif crop_stage == 2:  # Seedling
        target_depth = params.SEEDLING_IDEAL_DEPTH
        min_depth    = params.SEEDLING_MIN_DEPTH
        max_depth    = params.SEEDLING_MAX_DEPTH
    elif crop_stage == 3:  # Vegetative
        target_depth = params.VEGETATIVE_IDEAL_DEPTH
        min_depth    = params.VEGETATIVE_MIN_DEPTH
        max_depth    = params.VEGETATIVE_MAX_DEPTH
    elif crop_stage == 4:  # Reproductive
        target_depth = params.REPRODUCTIVE_IDEAL_DEPTH
        min_depth    = params.REPRODUCTIVE_MIN_DEPTH
        max_depth    = params.REPRODUCTIVE_MAX_DEPTH
    else:                  # Ripening (stage 5)
        target_moisture = params.RIPENING_IDEAL_MOISTURE
        min_moisture    = params.RIPENING_MIN_MOISTURE
        max_moisture    = params.RIPENING_MAX_MOISTURE

    # ========================================================================
    # STEP 2: CALCULATE BASE CWR USING WATER BALANCE
    # ========================================================================

    critical_situation = False   # Flag for urgent irrigation needs

    if crop_stage <= 4:   # FLOODED STAGES (Land Prep to Reproductive)

        current_depth_mm = water_depth_cm * 10   # Convert cm to mm

        # Below stage minimum = critical situation (Q1-Q4 min values)
        if current_depth_mm < min_depth:
            critical_situation = True

        if water_depth_cm < params.DRIED_FIELD_THRESHOLD:
            # DRIED FIELD: need to saturate soil first, then add standing water

            # Water needed to saturate soil from current moisture level
            if soil_moisture_pct < params.FLOODED_SOIL_SATURATION:
                soil_moisture_deficit = ((params.FLOODED_SOIL_SATURATION - soil_moisture_pct)
                                         * params.MOISTURE_TO_WATER_FACTOR)
            else:
                soil_moisture_deficit = 0

            # Full target standing water depth + soil saturation + losses
            base_CWR = (soil_moisture_deficit + target_depth +
                        ET_mm + params.PERCOLATION_RATE - rainfall_mm)

        else:
            # NORMAL FLOODED CONDITION — soil already saturated
            depth_deficit = max(0, target_depth - current_depth_mm)
            base_CWR = (depth_deficit + ET_mm + params.PERCOLATION_RATE - rainfall_mm)

    else:   # RIPENING STAGE (Stage 5) — drained field

        # Below minimum moisture = critical situation (Q5 min value)
        if soil_moisture_pct < params.RIPENING_MIN_MOISTURE:
            critical_situation = True

        moisture_deficit = max(0, target_moisture - soil_moisture_pct)
        water_needed_for_moisture = moisture_deficit * params.MOISTURE_TO_WATER_FACTOR
        base_CWR = (water_needed_for_moisture + ET_mm +
                    params.PERCOLATION_RATE - rainfall_mm)

    # ========================================================================
    # STEP 3: APPLY RAINFALL DECISION RULES (Q10)
    # ========================================================================

    if rainfall_mm > params.RAINFALL_SKIP_THRESHOLD:
        return 0   # Heavy rain — skip irrigation entirely

    elif rainfall_mm > params.RAINFALL_REDUCE_THRESHOLD:
        base_CWR = base_CWR * params.RAINFALL_REDUCTION_FACTOR   # Moderate rain — reduce to 40%

    # ========================================================================
    # STEP 4: CRITICAL SITUATION BOOST (Q1-Q4 min, Q5 min moisture)
    # ========================================================================

    if critical_situation:
        base_CWR = base_CWR * 1.3   # Boost 30% — restore conditions quickly

    # ========================================================================
    # STEP 5: TANK LEVEL CHECK (Q7, Q8)
    # ========================================================================
    # Q7: Tank min threshold is a binary operational gate —
    # below minimum, delivery pressure is unreliable regardless of CWR.
    # Above minimum, full CWR is delivered (tank level does not
    # gradually reduce irrigation — the CWR formula already scales output).
    
    if tank_level_pct >= params.TANK_MIN_THRESHOLD:
        # Sufficient tank level — deliver full calculated CWR
        tank_factor = 1.0
    
    else:
        # BELOW MINIMUM — apply stage priority (Q8)
        if crop_stage in params.PRIORITY_STAGES or critical_situation:
            tank_factor = 0.5   # Priority stages (Reproductive, Vegetative) get 50%
        else:
            tank_factor = 0.2   # Non-priority stages get 20%
    
    adjusted_CWR = base_CWR * tank_factor


    # ========================================================================
    # STEP 6: HIGH ET ADJUSTMENT (Q11, Q14, Q16)
    # ========================================================================

    if ET_mm > params.HIGH_ET_THRESHOLD:
        adjusted_CWR = adjusted_CWR * (1 + params.HIGH_ET_BUFFER)   # Add 15% buffer

    # ========================================================================
    # STEP 7: SAFETY CONSTRAINTS
    # ========================================================================

    if crop_stage <= 4:   # Flooded stages
        current_depth_mm = water_depth_cm * 10

        # Stage max depth IS the harmful threshold (Q5 removed, Q1-Q4 max used directly)
        max_addable = max_depth - current_depth_mm

        if max_addable < 0:
            return 0   # Field already over-flooded — no irrigation

        adjusted_CWR = min(adjusted_CWR, max_addable)

    # Q17: Minimum threshold check — with critical situation override
    if adjusted_CWR < params.MIN_CWR_THRESHOLD:
        if critical_situation:
            pass   # Emergency override — irrigate despite small amount
        else:
            adjusted_CWR = 0   # Too small — wait for deficit to accumulate

    # ========================================================================
    # STEP 8: FINAL OUTPUT
    # ========================================================================

    return max(0, adjusted_CWR)


# ============================================================================
# DATASET GENERATION FUNCTION
# ============================================================================

def generate_dataset(num_samples=1000, random_seed=42):
    """
    Generate synthetic training dataset for CWR prediction model.

    Parameters:
    -----------
    num_samples  : int — Number of training samples (default: 1000)
    random_seed  : int — Seed for reproducibility (default: 42)

    Returns:
    --------
    pandas.DataFrame : Dataset with input features and CWR labels
    """

    np.random.seed(random_seed)

    print(f"Generating {num_samples} training samples...")
    print("=" * 60)

    data = []
    samples_per_stage = num_samples // 5

    # ========================================================================
    # GENERATE SCENARIOS FOR EACH CROP STAGE
    # ========================================================================

    for stage in range(1, 6):   # Stages 1-5

        print(f"\nGenerating Stage {stage} ({STAGE_NAMES[stage]}) scenarios...")

        if stage <= 4:   # FLOODED STAGES

            if stage == 1:
                depth_min, depth_ideal, depth_max = (params.LAND_PREP_MIN_DEPTH / 10,
                                                      params.LAND_PREP_IDEAL_DEPTH / 10,
                                                      params.LAND_PREP_MAX_DEPTH / 10)
            elif stage == 2:
                depth_min, depth_ideal, depth_max = (params.SEEDLING_MIN_DEPTH / 10,
                                                      params.SEEDLING_IDEAL_DEPTH / 10,
                                                      params.SEEDLING_MAX_DEPTH / 10)
            elif stage == 3:
                depth_min, depth_ideal, depth_max = (params.VEGETATIVE_MIN_DEPTH / 10,
                                                      params.VEGETATIVE_IDEAL_DEPTH / 10,
                                                      params.VEGETATIVE_MAX_DEPTH / 10)
            else:   # stage == 4
                depth_min, depth_ideal, depth_max = (params.REPRODUCTIVE_MIN_DEPTH / 10,
                                                      params.REPRODUCTIVE_IDEAL_DEPTH / 10,
                                                      params.REPRODUCTIVE_MAX_DEPTH / 10)

            # Water depth distribution
            water_depths = []
            water_depths.extend(np.random.uniform(0, 0.4,
                                int(samples_per_stage * 0.20)))                   # 20% dried out
            water_depths.extend(np.random.uniform(depth_min, depth_ideal,
                                int(samples_per_stage * 0.30)))                   # 30% below ideal
            water_depths.extend(np.random.uniform(depth_ideal * 0.8, depth_ideal * 1.2,
                                int(samples_per_stage * 0.30)))                   # 30% around ideal
            water_depths.extend(np.random.uniform(depth_ideal, depth_max,
                                int(samples_per_stage * 0.20)))                   # 20% above ideal

            # Soil moisture — high when flooded, lower when dried
            soil_moistures = []
            for wd in water_depths:
                if wd < 0.5:
                    soil_moistures.append(np.random.uniform(50, 85))    # Dried field
                else:
                    soil_moistures.append(np.random.uniform(90, 100))   # Flooded field

        else:   # RIPENING STAGE (Stage 5) — drained field

            water_depths = np.zeros(samples_per_stage)   # No standing water

            soil_moistures = []
            soil_moistures.extend(np.random.uniform(params.RIPENING_MIN_MOISTURE,
                                                    params.RIPENING_IDEAL_MOISTURE,
                                                    int(samples_per_stage * 0.25)))   # 25% below ideal
            soil_moistures.extend(np.random.uniform(45, params.RIPENING_MIN_MOISTURE,
                                                    int(samples_per_stage * 0.25)))   # 25% critically dry
            soil_moistures.extend(np.random.uniform(params.RIPENING_IDEAL_MOISTURE * 0.9,
                                                    params.RIPENING_IDEAL_MOISTURE * 1.1,
                                                    int(samples_per_stage * 0.30)))   # 30% around ideal
            soil_moistures.extend(np.random.uniform(params.RIPENING_IDEAL_MOISTURE,
                                                    params.RIPENING_MAX_MOISTURE,
                                                    int(samples_per_stage * 0.20)))   # 20% above ideal

        water_depths   = list(water_depths)[:samples_per_stage]
        soil_moistures = list(soil_moistures)[:samples_per_stage]

        # Tank levels: uniform distribution
        tank_levels = np.random.uniform(15, 100, samples_per_stage)

        # ET rates: stage-specific distribution from agronomic literature
        high_prob   = params.ET_HIGH_PROB[stage]
        low_prob    = 0.10
        normal_prob = 1.0 - high_prob - low_prob

        ET_rates = []
        ET_rates.extend(np.random.uniform(params.ET_NORMAL_MIN, params.ET_NORMAL_MAX,
                                         int(samples_per_stage * normal_prob)))              # normal
        ET_rates.extend(np.random.uniform(params.HIGH_ET_THRESHOLD, params.ET_HIGH_MAX,
                                         int(samples_per_stage * high_prob)))                # high
        ET_rates.extend(np.random.uniform(4, params.ET_NORMAL_MIN,
                                         int(samples_per_stage * low_prob)))                 # low
        ET_rates = np.array(ET_rates[:samples_per_stage])
        np.random.shuffle(ET_rates)

        # Rainfall distribution (Q10)
        rainfalls = []
        rainfalls.extend(np.zeros(int(samples_per_stage * 0.50)))                           # 50% no rain
        rainfalls.extend(np.random.uniform(0.1, 5,  int(samples_per_stage * 0.25)))         # 25% light
        rainfalls.extend(np.random.uniform(5,   10, int(samples_per_stage * 0.15)))         # 15% moderate
        rainfalls.extend(np.random.uniform(10,  20, int(samples_per_stage * 0.10)))         # 10% heavy
        rainfalls = np.array(rainfalls[:samples_per_stage])
        np.random.shuffle(rainfalls)

        # Generate samples for this stage
        for i in range(samples_per_stage):

            water_depth   = water_depths[i]
            soil_moisture = soil_moistures[i]
            tank_level    = tank_levels[i]
            ET            = ET_rates[i]
            rainfall      = rainfalls[i]

            CWR = calculate_CWR(water_depth, soil_moisture, tank_level,
                                ET, rainfall, stage)

            data.append({
                'Water_Depth_cm':        round(water_depth, 2),
                'Soil_Moisture_%':       round(soil_moisture, 1),
                'Tank_Level_%':          round(tank_level, 1),
                'ET_mm_day':             round(ET, 2),
                'Rainfall_Predicted_mm': round(rainfall, 2),
                'Crop_Stage':            stage,
                'CWR_mm':                round(CWR, 2)
            })

        print(f"  Generated {samples_per_stage} samples for Stage {stage} ({STAGE_NAMES[stage]})")

    # ========================================================================
    # CREATE DATAFRAME
    # ========================================================================

    df = pd.DataFrame(data)
    df = df.sample(frac=1, random_state=random_seed).reset_index(drop=True)

    print("\n" + "=" * 60)
    print("Dataset generation complete!")
    print(f"Total samples: {len(df)}")
    print("\nDataset statistics:")
    print(df.describe())

    print("\nCWR distribution:")
    print(f"  Zero CWR (skip irrigation): {(df['CWR_mm'] == 0).sum()} samples ({(df['CWR_mm'] == 0).sum()/len(df)*100:.1f}%)")
    print(f"  Low CWR   (5-15mm):  {((df['CWR_mm'] >= 5)  & (df['CWR_mm'] < 15)).sum()} samples")
    print(f"  Moderate  (15-30mm): {((df['CWR_mm'] >= 15) & (df['CWR_mm'] < 30)).sum()} samples")
    print(f"  High CWR  (30-45mm): {((df['CWR_mm'] >= 30) & (df['CWR_mm'] <= 45)).sum()} samples")

    print("\nSamples per crop stage:")
    stage_counts = df['Crop_Stage'].value_counts().sort_index()
    for s, count in stage_counts.items():
        print(f"  Stage {s} ({STAGE_NAMES[s]}): {count} samples")

    # ========================================================================
    # MONOTONICITY SANITY CHECKS
    # ========================================================================

    print("\n" + "=" * 60)
    print("RUNNING MONOTONICITY SANITY CHECKS...")
    print("=" * 60)

    all_passed = True

    # Use Stage 3 (Vegetative) as reference — flooded, high ET stage
    ref = df[df['Crop_Stage'] == 3].copy()

    corr_rain = ref['CWR_mm'].corr(ref['Rainfall_Predicted_mm'])
    if corr_rain < 0:
        print(f"  Rainfall vs CWR (Stage 3 Vegetative): Negative ({corr_rain:.3f}) — PASSED")
    else:
        print(f"  Rainfall vs CWR (Stage 3 Vegetative): Expected negative ({corr_rain:.3f}) — FAILED")
        all_passed = False

    corr_et = ref['CWR_mm'].corr(ref['ET_mm_day'])
    if corr_et > 0:
        print(f"  ET vs CWR (Stage 3 Vegetative): Positive ({corr_et:.3f}) — PASSED")
    else:
        print(f"  ET vs CWR (Stage 3 Vegetative): Expected positive ({corr_et:.3f}) — FAILED")
        all_passed = False

    ref_s5  = df[df['Crop_Stage'] == 5].copy()
    corr_sm = ref_s5['CWR_mm'].corr(ref_s5['Soil_Moisture_%'])
    if corr_sm < 0:
        print(f"  Soil Moisture vs CWR (Stage 5 Ripening): Negative ({corr_sm:.3f}) — PASSED")
    else:
        print(f"  Soil Moisture vs CWR (Stage 5 Ripening): Expected negative ({corr_sm:.3f}) — FAILED")
        all_passed = False

    heavy_rain     = df[df['Rainfall_Predicted_mm'] > params.RAINFALL_SKIP_THRESHOLD]
    non_zero_heavy = (heavy_rain['CWR_mm'] > 0).sum()
    if non_zero_heavy == 0:
        print(f"  Heavy rain (>10mm) CWR=0: All {len(heavy_rain)} samples correct — PASSED")
    else:
        print(f"  Heavy rain (>10mm) CWR=0: {non_zero_heavy} samples have CWR>0 — FAILED")
        all_passed = False

    negative_cwr = (df['CWR_mm'] < 0).sum()
    if negative_cwr == 0:
        print(f"  No negative CWR values — PASSED")
    else:
        print(f"  Found {negative_cwr} negative CWR values — FAILED")
        all_passed = False

    print("\n" + "=" * 60)
    if all_passed:
        print("ALL SANITY CHECKS PASSED — Dataset is agronomically consistent")
    else:
        print("SOME CHECKS FAILED — Review CWR formula before training")
    print("=" * 60)

    return df


# ============================================================================
# MAIN EXECUTION
# ============================================================================

if __name__ == "__main__":

    print("\n" + "=" * 60)
    print("CWR DATASET GENERATOR FOR PADDY CULTIVATION")
    print("Machine Learning-Based Predictive Irrigation System")
    print("=" * 60)

    dataset = generate_dataset(num_samples=5000, random_seed=42)

    timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")
    filename  = f"paddy_CWR_dataset_{timestamp}.csv"
    dataset.to_csv(filename, index=False)
    print(f"\nDataset saved to: {filename}")

    print("\nFirst 10 samples:")
    print(dataset.head(10).to_string())

    print("\n" + "=" * 60)
    print("EXAMPLE CWR CALCULATIONS")
    print("=" * 60)

    examples = [
        {
            'desc': 'Land Preparation — Normal flooding',
            'water_depth': 5.0, 'soil_moisture': 93, 'tank': 70,
            'ET': 4, 'rain': 0, 'stage': 1
        },
        {
            'desc': 'Seedling — Adequate water',
            'water_depth': 4.0, 'soil_moisture': 95, 'tank': 70,
            'ET': 5, 'rain': 0, 'stage': 2
        },
        {
            'desc': 'Vegetative — Dried field, critical',
            'water_depth': 0.2, 'soil_moisture': 55, 'tank': 80,
            'ET': 7, 'rain': 0, 'stage': 3
        },
        {
            'desc': 'Reproductive — Heavy rain coming',
            'water_depth': 7.0, 'soil_moisture': 96, 'tank': 75,
            'ET': 6, 'rain': 12, 'stage': 4
        },
        {
            'desc': 'Ripening — Dry soil, critical',
            'water_depth': 0, 'soil_moisture': 48, 'tank': 65,
            'ET': 5, 'rain': 0, 'stage': 5
        },
        {
            'desc': 'Vegetative — High ET, low tank',
            'water_depth': 5.5, 'soil_moisture': 93, 'tank': 35,
            'ET': 9, 'rain': 0, 'stage': 3
        }
    ]

    for ex in examples:
        CWR = calculate_CWR(ex['water_depth'], ex['soil_moisture'], ex['tank'],
                            ex['ET'], ex['rain'], ex['stage'])
        print(f"\n{ex['desc']}:")
        print(f"  Inputs: Depth={ex['water_depth']}cm, Moisture={ex['soil_moisture']}%, "
              f"Tank={ex['tank']}%, ET={ex['ET']}mm, Rain={ex['rain']}mm, "
              f"Stage={ex['stage']} ({STAGE_NAMES[ex['stage']]})")
        print(f"  CWR Output: {CWR:.2f} mm")

    print("\n" + "=" * 60)
    print("Script execution completed successfully!")
    print("=" * 60 + "\n")


CWR DATASET GENERATOR FOR PADDY CULTIVATION
Machine Learning-Based Predictive Irrigation System
Generating 5000 training samples...

Generating Stage 1 (Land Preparation) scenarios...
  Generated 1000 samples for Stage 1 (Land Preparation)

Generating Stage 2 (Seedling) scenarios...
  Generated 1000 samples for Stage 2 (Seedling)

Generating Stage 3 (Vegetative) scenarios...
  Generated 1000 samples for Stage 3 (Vegetative)

Generating Stage 4 (Reproductive) scenarios...
  Generated 1000 samples for Stage 4 (Reproductive)

Generating Stage 5 (Ripening) scenarios...
  Generated 1000 samples for Stage 5 (Ripening)

Dataset generation complete!
Total samples: 5000

Dataset statistics:
       Water_Depth_cm  Soil_Moisture_%  Tank_Level_%    ET_mm_day  \
count     5000.000000       5000.00000    5000.00000  5000.000000   
mean         5.405524         88.34992      57.77266     6.655820   
std          6.096307         12.53548      24.75249     1.648958   
min          0.000000         45

In [1]:
from sklearn.metrics import cohen_kappa_score
import numpy as np

# Paste the 30 ratings from each expert (numbers 1–5 only)
# Must be in the same Sample ID order for both

farmer_ratings = [4,4,5,4,4,5,4,5,5,5,4,5,4,4,4,4,5,5,4,5,4,4,4,5,5,5,5,5,4,5]
rrdi_ratings   = [4,4,4,4,4,5,4,4,3,4,3,5,4,3,4,4,3,4,3,4,4,4,4,4,4,4,4,4,3,5]

# Weighted Kappa (linear weights — correct for ordinal Likert scale)
kappa = cohen_kappa_score(farmer_ratings, rrdi_ratings, weights='linear')

# Mean scores
all_ratings = farmer_ratings + rrdi_ratings
mean_score = np.mean(all_ratings)
pct_above_4 = sum(r >= 4 for r in all_ratings) / len(all_ratings) * 100

print(f"Weighted Kappa:          κ = {kappa:.3f}")
print(f"Mean plausibility score: {mean_score:.2f} / 5")
print(f"Rated 4 or 5:            {pct_above_4:.1f}%")

Weighted Kappa:          κ = 0.143
Mean plausibility score: 4.20 / 5
Rated 4 or 5:            90.0%


In [2]:
from sklearn.metrics import cohen_kappa_score
import numpy as np

# Paste the 30 ratings from each expert (numbers 1–5 only)
# Must be in the same Sample ID order for both

farmer_ratings = [5,5,5,5,5,5,5,5,5,5,5,5,5,4,4,4,5,5,5,4,4,4,4,5,5,5,5,5,5,5]

rrdi_ratings   = [4,4,4,4,2,3,4,4,2,4,2,4,3,3,3,4,3,4,3,4,4,3,4,4,2,2,2,2,2,4]

# Weighted Kappa (linear weights — correct for ordinal Likert scale)
kappa = cohen_kappa_score(farmer_ratings, rrdi_ratings, weights='linear')

# Mean scores
all_ratings = farmer_ratings + rrdi_ratings
mean_score = np.mean(all_ratings)
pct_above_4 = sum(r >= 4 for r in all_ratings) / len(all_ratings) * 100

print(f"Weighted Kappa:          κ = {kappa:.3f}")
print(f"Mean plausibility score: {mean_score:.2f} / 5")
print(f"Rated 4 or 5:            {pct_above_4:.1f}%")

Weighted Kappa:          κ = 0.000
Mean plausibility score: 4.00 / 5
Rated 4 or 5:            75.0%


In [4]:
from sklearn.metrics import cohen_kappa_score
import numpy as np

# ============================================================
# EXPERT RATINGS — 30 samples each (same order)
# ============================================================

farmer_ratings = [4,4,5,4,4,5,4,5,5,5,4,5,4,4,4,4,5,5,4,5,4,4,4,5,5,5,5,5,4,5]
rrdi_ratings   = [4,4,4,4,4,5,4,4,3,4,3,5,4,3,4,4,3,4,3,4,4,4,4,4,4,4,4,4,3,5]

# ============================================================
# METRIC 1 — Mean Plausibility Score
# ============================================================

all_ratings = farmer_ratings + rrdi_ratings
mean_score = np.mean(all_ratings)
pct_above_4 = sum(r >= 4 for r in all_ratings) / len(all_ratings) * 100

print("=" * 55)
print("METRIC 1 — Mean Plausibility Score")
print("=" * 55)
print(f"  Mean score:       {mean_score:.2f} / 5")
print(f"  Rated 4 or 5:     {pct_above_4:.1f}%")

# ============================================================
# METRIC 2 — Percentage Agreement Within 1 Point
# ============================================================

agreed = sum(abs(f - r) <= 1 for f, r in zip(farmer_ratings, rrdi_ratings))
pct_agreement = agreed / len(farmer_ratings) * 100

print("\n" + "=" * 55)
print("METRIC 2 — Percentage Agreement Within 1 Point")
print("=" * 55)
print(f"  Samples agreed (±1): {agreed} / {len(farmer_ratings)}")
print(f"  Agreement rate:      {pct_agreement:.1f}%")

# ============================================================
# METRIC 3 — Content Validity Index (CVI)
# ============================================================

cvi_scores = [1 if (f >= 4 and r >= 4) else 0
              for f, r in zip(farmer_ratings, rrdi_ratings)]
CVI = sum(cvi_scores) / len(cvi_scores)

print("\n" + "=" * 55)
print("METRIC 3 — Content Validity Index (CVI)")
print("=" * 55)
print(f"  Samples rated ≥4 by both: {sum(cvi_scores)} / {len(cvi_scores)}")
print(f"  CVI:                      {CVI:.3f}")
print(f"  Threshold (≥0.80):        {'PASSED' if CVI >= 0.80 else 'FAILED'}")

# ============================================================
# METRIC 4 — Per Stage Breakdown
# ============================================================

stage_names = ['Land Preparation', 'Seedling', 'Vegetative',
               'Reproductive', 'Ripening']

print("\n" + "=" * 55)
print("METRIC 4 — Per Stage Combined Mean Score")
print("=" * 55)
for i, stage in enumerate(stage_names):
    s, e = i * 6, i * 6 + 6
    combined = np.mean(farmer_ratings[s:e] + rrdi_ratings[s:e])
    farmer_m = np.mean(farmer_ratings[s:e])
    rrdi_m   = np.mean(rrdi_ratings[s:e])
    print(f"  {stage:<20} Farmer={farmer_m:.2f}  RRDI={rrdi_m:.2f}  Combined={combined:.2f}")

# ============================================================
# SUMMARY
# ============================================================

print("\n" + "=" * 55)
print("SUMMARY")
print("=" * 55)
print(f"  Mean plausibility score : {mean_score:.2f} / 5")
print(f"  Rated 4 or 5            : {pct_above_4:.1f}%")
print(f"  Agreement within 1 pt   : {pct_agreement:.1f}%")
print(f"  Content Validity Index  : {CVI:.3f}")

METRIC 1 — Mean Plausibility Score
  Mean score:       4.20 / 5
  Rated 4 or 5:     90.0%

METRIC 2 — Percentage Agreement Within 1 Point
  Samples agreed (±1): 28 / 30
  Agreement rate:      93.3%

METRIC 3 — Content Validity Index (CVI)
  Samples rated ≥4 by both: 24 / 30
  CVI:                      0.800
  Threshold (≥0.80):        PASSED

METRIC 4 — Per Stage Combined Mean Score
  Land Preparation     Farmer=4.33  RRDI=4.17  Combined=4.25
  Seedling             Farmer=4.67  RRDI=3.83  Combined=4.25
  Vegetative           Farmer=4.33  RRDI=3.67  Combined=4.00
  Reproductive         Farmer=4.33  RRDI=3.83  Combined=4.08
  Ripening             Farmer=4.83  RRDI=4.00  Combined=4.42

SUMMARY
  Mean plausibility score : 4.20 / 5
  Rated 4 or 5            : 90.0%
  Agreement within 1 pt   : 93.3%
  Content Validity Index  : 0.800
